# ARIM-Academy：ケモメトリクス（教師なし）～有機半導体ラマンスペクトルのクラスタリング～

## 対象読者・前提知識・動作環境

| 項目 | 内容 |
| --- | --- |
| **対象読者** | `RamanSPy-1`（前処理入門）を終えた方。Pythonの基礎文法は理解しているが、多変量解析（クラスタリング・PCA）に初めて触れる研究者・技術者 |
| **前提知識** | ラマン分光の基礎、`RamanSPy-1` の前処理。統計・機械学習の予備知識は不要 |
| **動作環境** | Python 3.10 以上、`ramanspy`, `numpy`, `pandas`, `scikit-learn`, `scipy`, `seaborn`, `matplotlib`, `adjustText`。Google Colab 推奨 |
| **版・ライセンス** | 本教材のコードは MIT License。データセット OS-127 の利用条件は「データセット」節を参照 |

本ノートブックは、OS-127 の**45本の有機材料ラマンスペクトルを一括前処理してデータ行列を作り、教師なし学習（階層クラスタリング・主成分分析・k-means法）でスペクトルの類似構造を可視化・解釈する**ことを目的とします。教師あり分類（材料同定・分類モデルの比較）は続編 `RamanSPy-3` で扱います。

## 本編の目標：ラマンスペクトルを用いたケモメトリクス（教師なし）

この講義では、OS-127「有機色素ラマン分光データセット」を用いて、ラベル（正解）を使わずにスペクトルの類似構造を探る**ケモメトリクス**を実践します。具体的には次を学びます。

1. **ラマン分光データの一括前処理**：RamanSPy のパイプライン機能で、45本のスペクトルに同一の前処理（クロッピング・宇宙線除去・平滑化・ベースライン補正・規格化）を適用します。
2. **共通波数軸へのリサンプリング**：測定条件（励起波長・回折格子）が異なるため、スペクトルごとに波数軸が異なります。多変量解析に必要な「そろったデータ行列」を作るために、共通の波数グリッドへ内挿（リサンプリング）します。
3. **教師なし学習の適用**：階層クラスタリング分析（HCA）、主成分分析（PCA）、k-means法を用いて、スペクトルのグループ構造を可視化し、その化学的な意味を考察します。

**キーメッセージ**：ラマンスペクトルの類似度は、材料の「デバイス上の役割（ドナー／アクセプター／輸送層…）」よりも「**分子構造（骨格の化学）**」を強く反映します。教師なし学習が見つけるグループが何を意味するのかを、実データで確かめます。

## データセット：OS-127 有機色素ラマン分光データセット

<div style="border:1px solid #000; padding:10px;">

**ARIM（マテリアル先端リサーチインフラ）事業**で大阪大学が公開した **OS-127「有機色素ラマン分光データセット」** を用います。有機薄膜太陽電池（OPV）・有機EL（OLED）・有機電界効果トランジスタ（OFET）などで重要な**45本のスペクトル（41種の材料）**を、レーザーラマン顕微鏡（Nanophoton RAMANtouch VIS-NIR-OUN、励起波長 532 / 785 nm）で測定した参照スペクトル集です。

各材料には、有機エレクトロニクスでの役割に基づく **5つの機能カテゴリ（family）** を付与してあります（`data/OS-127_labels.csv`）。

| family | 日本語 | 例 | 本数 |
| --- | --- | --- | --- |
| `donor_polymer` | ドナー高分子 | PM6, PTB7, D18, PTQ10 | 11 |
| `acceptor` | 非フラーレンアクセプター | Y6, ITIC, IEICO-4F, PDIN | 10 |
| `hole_transport` | 正孔輸送・注入材料 | Spiro-MeOTAD, PEDOT:PSS, 2PACz | 8 |
| `electron_transport` | 電子輸送・中間層材料 | BCP, Alq3, PFN-Br | 6 |
| `molecular_dye` | 色素・低分子半導体 | CuPc, Rubrene, Methylene Blue | 10 |

これらのカテゴリは「正解ラベル」ではなく、**教師なし学習が見つけたグループと見比べるための参照**として使います（このノートブックのクラスタリングはカテゴリを一切使いません）。

- 提供機関：大阪大学（ARIM事業、課題番号 JPMXP1226OS9001）／装置：Nanophoton RAMANtouch VIS-NIR-OUN

</div>

# Google Colabにおける環境設定
Google Colab 環境でなければ実行不要です。

In [ ]:
!pip install ramanspy
!pip install adjustText
!git clone https://github.com/ARIM-ACADEMY-2026/Advanced_Tutorial_8_RamanSPy.git
%cd Advanced_Tutorial_8_RamanSPy

### ライブラリのインポート
本ノートブックで使うライブラリを読み込みます。再現性のため乱数シードを固定し、生成物の保存先 `output/` を用意します。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ラマンの前処理
import ramanspy as rp

from warnings import filterwarnings
filterwarnings("ignore")

# 再現性のための乱数シード固定
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 入出力パス
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data" / "spectra"
LABEL_CSV = BASE_DIR / "data" / "OS-127_labels.csv"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. 事前準備 ～ユーザー定義関数～

### CSV読み取り関数
OS-127 のCSV（`Wavenumber, Intensity` の2列）を RamanSPy の `Spectrum` オブジェクトに変換する関数です。

In [ ]:
def read_csv(csv_filename):
    """OS-127 形式のCSVを RamanSPy の Spectrum オブジェクトに変換する。

    @param csv_filename : 入力CSVファイルのパス
    @return : rp.Spectrum オブジェクト
    """
    data = pd.read_csv(csv_filename)
    wavenumber_axis = data["Wavenumber"].values
    intensity = data["Intensity"].values
    return rp.Spectrum(intensity, wavenumber_axis)

### 前処理パイプライン関数
`RamanSPy-1` で学んだ前処理を1本のパイプラインにまとめ、全スペクトルに同一適用します。順序は「クロッピング → 宇宙線除去 → Savitzky-Golay 平滑化 → ベースライン補正（asPLS） → Min-max 規格化」です。

In [ ]:
# クロッピング範囲（各スペクトルの測定範囲内で共通に取れる指紋領域）
CROP_REGION = (200, 1800)
# Savitzky-Golay フィルタ
SG_WINDOW_LENGTH = 15
SG_POLYORDER = 3

def preprocess(spectrum):
    """1本のスペクトルに前処理パイプラインを適用して返す。"""
    pipeline = rp.preprocessing.Pipeline([
        rp.preprocessing.misc.Cropper(region=CROP_REGION),
        rp.preprocessing.despike.WhitakerHayes(kernel_size=5),
        rp.preprocessing.denoise.SavGol(window_length=SG_WINDOW_LENGTH, polyorder=SG_POLYORDER),
        rp.preprocessing.baseline.ASPLS(),
        rp.preprocessing.normalise.MinMax(),
    ])
    return pipeline.apply(spectrum)

### 共通波数グリッドへのリサンプリング関数
**実データ特有の注意点**：OS-127 の各スペクトルは、励起波長や回折格子（300 / 600 / 1200 gr/mm）が異なるため、波数軸の範囲・刻み幅・点数がバラバラです。多変量解析では全サンプルが**同じ列（同じ波数点）**を持つデータ行列が必要なので、共通の波数グリッドへ内挿（1次元補間）してそろえます。

ここでは、45本中ほとんどのスペクトルが測定範囲に含む **900–1600 cm⁻¹** を共通の指紋領域とし、2 cm⁻¹ 刻みのグリッドに内挿します。この範囲を測定していないスペクトルは、外挿による捏造を避けるため解析から除外します（これも実データ解析では避けて通れない判断です）。

In [ ]:
# 共通波数グリッド（指紋領域）
GRID_LOW, GRID_HIGH, GRID_STEP = 900.0, 1600.0, 2.0
COMMON_GRID = np.arange(GRID_LOW, GRID_HIGH + GRID_STEP, GRID_STEP)

def resample_to_grid(spectrum, grid=COMMON_GRID):
    """前処理済みスペクトルを共通波数グリッドへ内挿する。
    グリッド範囲を測定範囲が覆っていない場合は None を返す（外挿しない）。
    """
    axis = np.asarray(spectrum.spectral_axis, dtype=float)
    values = np.asarray(spectrum.spectral_data, dtype=float)
    order = np.argsort(axis)
    axis, values = axis[order], values[order]
    if axis.min() > grid.min() or axis.max() < grid.max():
        return None
    return np.interp(grid, axis, values)

### 保存関数
前処理済みスペクトルを `output/` にCSV保存する関数です。

In [ ]:
HEADER = "Wavenumber,Intensity"

def dump_csv(output_csv, spectrum):
    """前処理済みスペクトルを2列（波数・強度）のCSVに保存する。"""
    x = spectrum.spectral_axis
    y = spectrum.spectral_data
    np.savetxt(output_csv, np.c_[x, y], delimiter=",", header=HEADER, comments="")

## 2. スペクトルデータの読み込み

### ファイル一覧とラベルの読み込み
`data/spectra/` 内の全CSVを取得し、材料名・機能カテゴリ（family）・測定条件をまとめた `OS-127_labels.csv` を読み込みます。

In [ ]:
files = sorted(DATA_DIR.glob("*.csv"))
print(f"スペクトル本数: {len(files)}")

labels = pd.read_csv(LABEL_CSV).set_index("file")
labels.head()

## 3. 前処理と共通グリッドへのリサンプリング

### 一括前処理・リサンプリング・データ行列の作成
各スペクトルを読み込み、前処理パイプラインを適用し、前処理済みCSVを保存し、共通グリッドへ内挿してデータ行列を組み立てます。共通グリッド（900–1600 cm⁻¹）を覆わないスペクトルは除外し、その旨を表示します。

In [ ]:
%%time
data_rows = []
sample_names = []
excluded = []

for csvfile in files:
    spectrum = read_csv(csvfile)
    corrected = preprocess(spectrum)

    # 前処理済みスペクトルを保存
    dump_csv(OUTPUT_DIR / (csvfile.stem + "_corrected.csv"), corrected)

    # 共通グリッドへリサンプリング
    resampled = resample_to_grid(corrected)
    if resampled is None:
        excluded.append(csvfile.name)
        continue

    data_rows.append(resampled)
    sample_names.append(csvfile.name)

print(f"データ行列に採用: {len(sample_names)} 本")
print(f"除外（900–1600 cm⁻¹ を測定範囲が覆わない）: {len(excluded)} 本 -> {excluded}")

### データ行列

In [ ]:
# 行=スペクトル、列=共通波数グリッド
df = pd.DataFrame(data_rows, index=sample_names, columns=np.round(COMMON_GRID, 1))

# 各スペクトルに機能カテゴリ（family）を対応づける（可視化の色分け用。クラスタリングには使わない）
family = labels.loc[df.index, "family"]
print("データ行列の形状:", df.shape)
print("\nfamily 別の本数:")
print(family.value_counts())
df.iloc[:, :5]

### 前処理済みスペクトルの重ね描き
データ行列の中身を、family ごとに色分けして重ね描きします（**図1**）。同じカテゴリでもスペクトル形状はかなり異なり、逆に別カテゴリでも似た形状のものがあることに注目してください。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
palette = dict(zip(sorted(family.unique()), sns.color_palette("Set1", family.nunique())))

for name in df.index:
    ax.plot(df.columns.astype(float), df.loc[name].values,
            color=palette[family[name]], alpha=0.6, lw=0.9)

handles = [plt.Line2D([0], [0], color=c, label=f) for f, c in palette.items()]
ax.legend(handles=handles, title="family", fontsize=9)
ax.set_xlabel("Raman shift [cm$^{-1}$]", fontsize=13)
ax.set_ylabel("Normalised intensity [a.u.]", fontsize=13)
ax.set_title("Preprocessed spectra on the common grid (900-1600 cm$^{-1}$)")
ax.invert_xaxis()
ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig1_spectra_overlay.png", dpi=150)
plt.show()

## 4. 教師なし学習によるクラスタリング分析

前処理済みデータ行列 `df`（行=スペクトル、列=波数）を使って、ラベルを使わずにスペクトルの類似構造を探ります。

1. **階層クラスタリング分析（HCA）**：スペクトル間の距離に基づき階層構造（デンドログラム）を作り、どのスペクトルが近いかを可視化します。
2. **主成分分析（PCA）**：高次元（数百点）のスペクトルを少数の主成分に圧縮し、散布図で全体像を眺めます。同時に**ローディング**から、どの波数がスペクトルの違いを生んでいるかを読み取ります。
3. **k-means法**：PCAで圧縮した空間でスペクトルを k 個のクラスタに分けます。エルボー法でクラスタ数の目安を探ります。

### HCA・PCA・k-means 用のライブラリ

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

### 4.1 階層クラスタリング分析（HCA）

各スペクトルを1点とみなし、ウォード法（`method="ward"`：併合時のクラスタ内分散増加を最小化）で階層的に併合します。結果はデンドログラム（樹形図）で表示します。ラベル文字は、材料名の横に family を色で示します。

**図2**：ウォード法によるデンドログラム

In [ ]:
plt.figure(figsize=(9, 14))

linkage_matrix = linkage(df.values, method="ward")

dendro = dendrogram(
    linkage_matrix,
    labels=[f"{n[:-4]}" for n in df.index],
    orientation="right",
    color_threshold=0.7 * linkage_matrix[:, 2].max(),
)

# ラベル文字を family の色で塗る
ax = plt.gca()
for lbl in ax.get_ymajorticklabels():
    fname = lbl.get_text() + ".csv"
    if fname in family.index:
        lbl.set_color(palette[family[fname]])

handles = [plt.Line2D([0], [0], color=c, label=f, marker="s", ls="") for f, c in palette.items()]
plt.legend(handles=handles, title="family", fontsize=9, loc="lower right")
plt.xlabel("Distance (Ward)")
plt.title("Hierarchical clustering of OS-127 Raman spectra")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig2_dendrogram.png", dpi=150)
plt.show()

### 【解説】`linkage()` と `dendrogram()`

- `linkage(X, method="ward")`：データ行列 `X` の全サンプル間を階層的に併合し、併合の履歴（どのクラスタがどの距離で結合したか）を返します。`"ward"` のほか `"single"`, `"complete"`, `"average"` などがあります。
- `dendrogram(linkage_matrix, ...)`：併合履歴を樹形図として描画します。横軸（この図では距離）が小さいところで結ばれるほど、スペクトルが似ていることを意味します。

**読み取りのポイント**：デンドログラムの枝の低い位置（近距離）で結ばれるペアに注目します。OS-127 では、**分子構造が近い材料どうし**が近距離で結合する傾向があります（次のセルで具体的に確認します）。

### HCAクラスタと family の対応
デンドログラムを5クラスタで切り分け、各クラスタにどの材料・family が含まれるかを一覧します。

In [ ]:
K_HCA = 5
hca_labels = fcluster(linkage_matrix, K_HCA, criterion="maxclust")

hca_table = pd.DataFrame({
    "material": labels.loc[df.index, "material"].values,
    "family": family.values,
    "hca_cluster": hca_labels,
}, index=df.index)

for c in sorted(set(hca_labels)):
    members = hca_table[hca_table.hca_cluster == c]
    print(f"[cluster {c}]  n={len(members)}")
    for _, row in members.iterrows():
        print(f"    {row['material']:16s}  ({row['family']})")
    print()

ari_hca = adjusted_rand_score(family.values, hca_labels)
print(f"HCAクラスタ vs family の一致度 (Adjusted Rand Index): {ari_hca:.3f}")

### 【考察】クラスタは「役割」でなく「分子構造」を映す

HCA が近距離で結ぶペア・小クラスタは、デバイス上の役割（family）ではなく**分子骨格の化学**で説明できます。代表例：

- **ペリレンジイミド系アクセプター**（PDIN・PDINN）が同一クラスタ。両者はペリレンジイミド骨格を共有し、ラマン指紋がほぼ同型です。
- **カルバゾール系ホスホン酸SAM**（2PACz・MeO-2PACz）が同一クラスタ。同じカルバゾール骨格に由来します。
- **フタロシアニン／ポルフィリン等の縮環色素**（CuPc・F16CuPc・無金属フタロシアニン・ClMnTPP・メチレンブルー）が、縮環アクセプター（Y6・IEICO-4F）とともに1つのグループを作ります。いずれも強く共役した縮環π系で、環伸縮由来の強いバンドを共有します。

一方で、同じ `acceptor` でも構造の異なる PDIN 系（ペリレンジイミド）と Y6 系（A–D–A縮環）は別クラスタに分かれます。**Adjusted Rand Index（family との一致度）が低い**のはこのためで、「ラマンのクラスタ＝機能カテゴリ」ではないことを定量的に示しています。これは失敗ではなく、ラマン分光が**分子構造の指紋**であることの表れです。

### 4.2 主成分分析（PCA）

数百点の波数からなる高次元スペクトルを、情報をできるだけ保ったまま少数の主成分（互いに無相関な合成軸）へ圧縮します。まず寄与率（各主成分が説明する分散の割合）を確認し、第1・第2主成分の散布図で全体像を眺めます。

In [ ]:
pca = PCA(n_components=10, random_state=RANDOM_STATE)
scores = pca.fit_transform(df.values)

evr = pca.explained_variance_ratio_
print("各主成分の寄与率:")
for i, r in enumerate(evr[:6], 1):
    print(f"  PC{i}: {r:6.1%}")
print(f"  PC1+PC2 累積: {evr[:2].sum():.1%}")
print(f"  PC1-PC5 累積: {evr[:5].sum():.1%}")

**図3**：寄与率のスクリープロット（累積寄与率つき）

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
xs = np.arange(1, len(evr) + 1)
ax.bar(xs, evr, alpha=0.7, label="individual")
ax.plot(xs, np.cumsum(evr), "o-", color="crimson", label="cumulative")
ax.set_xlabel("Principal component")
ax.set_ylabel("Explained variance ratio")
ax.set_title("PCA scree plot")
ax.set_xticks(xs)
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig3_scree.png", dpi=150)
plt.show()

**図4**：PC1–PC2 スコア散布図（family で色分け）
family は色分けの参照に使うだけで、PCA自体はラベルを使っていません。

In [ ]:
pca_df = pd.DataFrame(scores[:, :2], columns=["PC1", "PC2"], index=df.index)
pca_df["family"] = family.values

fig, ax = plt.subplots(figsize=(8, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="family",
                palette=palette, s=70, ax=ax)
ax.set_xlabel(f"PC1 ({evr[0]:.1%})")
ax.set_ylabel(f"PC2 ({evr[1]:.1%})")
ax.set_title("PCA score plot (colored by family)")
ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig4_pca_scores.png", dpi=150)
plt.show()

### ローディングの可視化
**ローディング**（主成分係数）は、各波数が主成分にどれだけ寄与するかを表します。ピークとして立っている波数が、その主成分方向でスペクトルを分けている「効いている振動モード」です。

**図5**：PC1・PC2 のローディングスペクトル

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(COMMON_GRID, pca.components_[0], label=f"PC1 loading ({evr[0]:.1%})")
ax.plot(COMMON_GRID, pca.components_[1], label=f"PC2 loading ({evr[1]:.1%})", alpha=0.8)
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("Raman shift [cm$^{-1}$]")
ax.set_ylabel("Loading")
ax.set_title("PCA loadings (which wavenumbers drive PC1/PC2)")
ax.invert_xaxis()
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig5_pca_loadings.png", dpi=150)
plt.show()

### 4.3 k-means 法（エルボー法によるクラスタ数の検討）

k-means は、あらかじめ指定したクラスタ数 k にデータを分割し、各点を最も近い重心へ割り当てる手法です。適切な k を選ぶ目安として**エルボー法**を使います。クラスタ内誤差平方和（SSE, inertia）を k ごとに計算し、減少が鈍る「肘」を探します。次元削減後の PC1–PC5 空間で計算します。

In [ ]:
pca_5d = scores[:, :5]

sse = []
K_RANGE = range(1, 11)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(pca_5d)
    sse.append(km.inertia_)

**図6**：エルボー法（SSE vs クラスタ数）

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(list(K_RANGE), sse, marker="o")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("SSE (inertia)")
ax.set_title("Elbow method")
ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig6_elbow.png", dpi=150)
plt.show()

### k-means クラスタリングと可視化
エルボーの目安（ここでは k=5、family 数と同数）でクラスタリングし、PCA散布図上に色分けして表示します。

In [ ]:
K_MEANS = 5
kmeans = KMeans(n_clusters=K_MEANS, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(pca_5d)

pca_df["kmeans"] = kmeans_labels
ari_km = adjusted_rand_score(family.values, kmeans_labels)
print(f"k-means クラスタ vs family の一致度 (Adjusted Rand Index): {ari_km:.3f}")

**図7**：k-means クラスタ（ラベル注釈つき）
`adjustText` でラベルの重なりを避けて材料名を表示します。

In [ ]:
from adjustText import adjust_text

fig, ax = plt.subplots(figsize=(11, 9))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="kmeans",
                palette="Set2", s=80, ax=ax, legend="full")

texts = []
for name in pca_df.index:
    texts.append(ax.annotate(name[:-4], (pca_df.loc[name, "PC1"], pca_df.loc[name, "PC2"]),
                             fontsize=8, color="black"))
adjust_text(texts, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

ax.set_xlabel(f"PC1 ({evr[0]:.1%})")
ax.set_ylabel(f"PC2 ({evr[1]:.1%})")
ax.set_title("k-means clusters on PCA space")
ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig7_kmeans.png", dpi=150)
plt.show()

## 5. 手法の比較（HCA vs k-means）

同じデータに対する2つの教師なし手法の結果を、family との一致度（Adjusted Rand Index; ARI）で横並びに比較します。ARIは0付近が「ランダムな一致」、1が「完全一致」です。

**図8**：HCA と k-means の family 一致度（ARI）比較

In [ ]:
compare = pd.DataFrame({
    "method": ["HCA (Ward)", "k-means (PC1-5)"],
    "n_clusters": [K_HCA, K_MEANS],
    "ARI_vs_family": [ari_hca, ari_km],
})
print(compare.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(compare["method"], compare["ARI_vs_family"], color=["steelblue", "darkorange"])
ax.set_ylabel("Adjusted Rand Index vs family")
ax.set_ylim(0, 1)
ax.set_title("Clustering agreement with device-function family")
for i, v in enumerate(compare["ARI_vs_family"]):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center")
ax.grid(True, axis="y")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig8_method_comparison.png", dpi=150)
plt.show()

### 【考察】比較から分かること

- HCA・k-means のいずれも、family（デバイス上の役割）との一致度（ARI）は低い水準にとどまります。これは手法の失敗ではなく、**ラマンスペクトルの類似度が分子構造に支配され、デバイス機能とは別軸**であることの反映です。
- 一方で、両手法とも**分子構造が近い材料**（ペリレンジイミド系、カルバゾールSAM系、縮環色素系など）は安定して同じグループに集めます。教師なし学習は「未知試料が既知のどの構造タイプに近いか」を探る用途に適しています。
- k-means は事前に k を決める必要があり、PCA前処理や乱数初期値（`random_state`）に結果が依存します。HCA は距離のしきい値で柔軟に粒度を変えられます。目的に応じて使い分けます。

## 6. まとめ

本ノートブックでは、OS-127 の有機材料ラマンスペクトルを対象に、教師なしケモメトリクスの一連の流れを実践しました。

- **一括前処理**：RamanSPy の `Pipeline` で45本に同一前処理を適用。
- **共通グリッド化**：測定条件で波数軸が異なる実データを、内挿で共通グリッドにそろえてデータ行列を作成（3本は範囲外のため除外）。
- **HCA・PCA・k-means**：スペクトルの類似構造を可視化。クラスタは分子構造を反映し、デバイス機能カテゴリ（family）とは一致度が低いことを ARI で定量的に確認。

**得られた知見**：ラマン分光＋教師なし学習は、材料の「構造タイプの発見・グルーピング」に有効。「機能で分類したい」場合は、構造情報だけでは足りず、教師あり学習や追加特徴量が必要になる（続編 `RamanSPy-3` へ）。

## 7. 本ノートブックで扱っていないこと（今後の課題）

- **教師あり分類・材料同定**：正解ラベルを使った分類モデルは続編 `RamanSPy-3` で扱います。
- **前処理・グリッド範囲の感度分析**：クロッピング範囲や共通グリッド（900–1600 cm⁻¹）の取り方でクラスタ構造がどう変わるかは検証していません。
- **クラスタ数の客観的決定**：エルボー法は主観的です。シルエット係数・ギャップ統計などの併用は扱っていません。
- **除外スペクトルの救済**：共通グリッド外の3本をどう組み込むか（領域を分ける、欠測扱いにする等）。

## 8. 演習問題

1. **クラスタリング手法の変更**：`linkage()` の `method` を `"ward"` から `"average"` や `"complete"` に変え、デンドログラムがどう変わるか比較してください。ペリレンジイミド系（PDIN/PDINN）の近接関係は保たれますか。
2. **主成分数の検討**：k-means を PC1–PC2 のみ、PC1–PC5、全次元（`df.values`）で行い、ARI と散布図がどう変わるか比較してください。
3. **共通グリッドの範囲**：`GRID_LOW`/`GRID_HIGH` を 1000–1600 に変えると、採用本数（除外本数）とクラスタ構造はどう変わりますか。
4. **ローディングの解釈**：PC1・PC2 のローディングでピークが立つ波数を読み取り、どの材料群を分けているか（図4のスコアと照合して）考察してください。
5. **励起波長の影響**：`OS-127_labels.csv` の `excitation_nm`（532 / 785 nm）で色分けした散布図を描き、クラスタ構造が材料構造由来か測定条件由来かを検討してください。